01. Setup

In [1]:
# Instalação do Java e Spark
!apt-get update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

# Variáveis de ambiente
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

# Pacote Iceberg
!wget -q https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.6.1/iceberg-spark-runtime-3.5_2.12-1.6.1.jar -P /content/spark-3.5.0-bin-hadoop3/jars/

import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col

# Sessão Spark
spark = SparkSession.builder \
    .appName("IcebergWithSpark") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.6.1,org.postgresql:postgresql:42.3.1") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hadoop_catalog.type", "hadoop") \
    .config("spark.sql.catalog.hadoop_catalog.warehouse", "/content/warehouse") \
    .config("spark.sql.default.catalog", "hadoop_catalog") \
    .getOrCreate()

# Diretório do Warehouse
!mkdir -p /content/warehouse

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [38.8 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,311 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [

In [9]:
from pyspark.sql.functions import to_date, col

### Compactação de Dados

In [10]:
# Exclui se existir
spark.sql("DROP TABLE IF EXISTS hadoop_catalog.default.vendas")

# Cria a tabela Vendas
spark.sql("""
    CREATE TABLE hadoop_catalog.default.vendas (
        id INT,
        produto STRING,
        quantidade INT,
        preco DOUBLE,
        data_venda DATE
    )
    USING iceberg
""")

DataFrame[]

In [11]:
# Inserir dados fracionados para criar vários arquivos

def inserir_dados(pequeno_lote):
    df = spark.createDataFrame(pequeno_lote, ["id", "produto", "quantidade", "preco", "data_venda"])
    df = df.withColumn("data_venda", to_date(col("data_venda"), "yyyy-MM-dd"))
    df.writeTo("hadoop_catalog.default.vendas").append()

# 10 Lotes de Dados
for i in range(1, 11):
    data = [
        (i, f"Produto {i}", i * 2, i * 10.0, f"2024-11-{i:02d}")
    ]
    inserir_dados(data)

In [12]:
# Contar arquivos e registros
print("Antes da compactação:")
df_files_before = spark.sql("""
    SELECT
        COUNT(*) AS registros,
        input_file_name() AS arquivo
    FROM hadoop_catalog.default.vendas
    GROUP BY input_file_name()
""")
df_files_before.show(truncate=False)


num_arquivos_antes = df_files_before.count()
print(f"Número total de arquivos: {num_arquivos_antes}")


Antes da compactação:
+---------+----------------------------------------------------------------------------------------------------+
|registros|arquivo                                                                                             |
+---------+----------------------------------------------------------------------------------------------------+
|1        |/content/warehouse/default/vendas/data/00001-11-1009a4a6-0603-4a8d-b167-23b86adf1a74-0-00001.parquet|
|1        |/content/warehouse/default/vendas/data/00001-23-bca34228-81b7-4545-a524-87dcbfa2f73d-0-00001.parquet|
|1        |/content/warehouse/default/vendas/data/00001-9-16e50cdc-f4a3-4aa0-9a8e-2747316105b7-0-00001.parquet |
|1        |/content/warehouse/default/vendas/data/00001-17-6bad19ff-adcb-4ced-89db-cda3d8e9040f-0-00001.parquet|
|1        |/content/warehouse/default/vendas/data/00001-13-c54b43a2-7ee3-4407-9ed4-a63f083a6bf4-0-00001.parquet|
|1        |/content/warehouse/default/vendas/data/00001-7-6e387147-741e-40

In [13]:
# Tamanho máximo de registros por arquivo
spark.conf.set("spark.sql.files.maxRecordsPerFile", 1000)

# Compactação com proc 'rewrite_data_files'
spark.sql("""
    CALL hadoop_catalog.system.rewrite_data_files(
        table => 'default.vendas'
    )
""")

DataFrame[rewritten_data_files_count: int, added_data_files_count: int, rewritten_bytes_count: bigint, failed_data_files_count: int]

In [14]:
# Contar arquivos e registros
print("Após a compactação:")
df_files_after = spark.sql("""
    SELECT
        COUNT(*) AS registros,
        input_file_name() AS arquivo
    FROM hadoop_catalog.default.vendas
    GROUP BY input_file_name()
""")
df_files_after.show(truncate=False)

num_arquivos_depois = df_files_after.count()
print(f"Número total de arquivos após a compactação: {num_arquivos_depois}")

Após a compactação:
+---------+----------------------------------------------------------------------------------------------------+
|registros|arquivo                                                                                             |
+---------+----------------------------------------------------------------------------------------------------+
|10       |/content/warehouse/default/vendas/data/00000-35-7ac468e4-860e-4204-9d76-8d6f7c49616e-0-00001.parquet|
+---------+----------------------------------------------------------------------------------------------------+

Número total de arquivos após a compactação: 1


In [15]:
# definindo período de retenação
spark.sql("""
    CALL hadoop_catalog.system.expire_snapshots(
        table => 'default.vendas',
        retain_last => 1
    )
""")

DataFrame[deleted_data_files_count: bigint, deleted_position_delete_files_count: bigint, deleted_equality_delete_files_count: bigint, deleted_manifest_files_count: bigint, deleted_manifest_lists_count: bigint, deleted_statistics_files_count: bigint]